# Building models with torch.nn: nn.Module, layers, and parameters

_PyTorch_

**Subclass nn.Module, declare layers in __init__, wire them in forward, and PyTorch tracks every weight for you.**

An nn.Module bundles two things: the state (its weight tensors, called parameters) and
       the computation (how to turn an input into an output). You define both in one class.

       The pattern never changes: declare the layers in __init__, then describe the data
       flow in forward. PyTorch does the rest &mdash; it finds every parameter, tracks gradients,
       and lets you save, load, and move the whole model with one call.

---

This notebook is a practice scaffold for the **Building models with torch.nn: nn.Module, layers, and parameters** lesson. We build the model up one step at a time — run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Define an MLP by subclassing nn.Module

The pattern never changes: declare the layers in `__init__` (so PyTorch **registers** them) and describe the data flow in `forward`. `super().__init__()` must come first — it sets up the registration machinery before any layer is assigned. The final layer emits raw logits, with no softmax inside the model.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)                      # reproducible weights and input

# Layers live in __init__ (so they are REGISTERED); the data flow lives in forward.
class MLP(nn.Module):
    def __init__(self, in_dim=784, hidden=256, out_dim=10):
        super().__init__()               # MUST be first: sets up registration
        self.fc1 = nn.Linear(in_dim, hidden)   # registered weight + bias
        self.relu = nn.ReLU()                  # parameter-free activation
        self.fc2 = nn.Linear(hidden, out_dim)  # registered weight + bias

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x                          # raw logits (no softmax here)

model = MLP()
print(model)                              # prints the module tree

### Step 2 — Inspect the registered parameters and run a forward pass

Because the layers were assigned to `self`, PyTorch auto-registers every weight as an autograd tensor — you can count them, see they default to `requires_grad=True`, and list the `state_dict` keys (what a checkpoint saves). To run the model, **call it** (`model(x)`), not `.forward(x)`: calling the module runs registered hooks plus the forward pass.

In [ ]:
# Parameters are auto-registered: each is an autograd tensor.
n_params = sum(p.numel() for p in model.parameters())
print("trainable parameters:", n_params)          # -> 203530
print("fc1.weight requires_grad:", model.fc1.weight.requires_grad)   # -> True
print("state_dict keys:", list(model.state_dict().keys()))
#   -> ['fc1.weight', 'fc1.bias', 'fc2.weight', 'fc2.bias']

# A forward pass on a random batch of 8 examples.
# CALL THE MODEL, not .forward(): model(x) runs hooks + forward.
x = torch.randn(8, 784)                   # batch of 8, 784 features each
logits = model(x)                         # NOT model.forward(x)
print("output shape:", logits.shape)      # -> torch.Size([8, 10])

### Step 3 — Build the same model with nn.Sequential

When the model is just a straight chain of layers, `nn.Sequential` stacks them without writing a class. It registers the same layers, so its parameter count and output shape are identical to the subclassed `MLP` above — `nn.Module` subclassing only earns its keep once the data flow branches.

In [ ]:
# The SAME MLP as nn.Sequential (quick stack, no class).
seq = nn.Sequential(
    nn.Linear(784, 256),
    nn.ReLU(),
    nn.Linear(256, 10),
)

seq_params = sum(p.numel() for p in seq.parameters())
print("sequential parameters:", seq_params)       # -> 203530 (identical)
print("sequential output:", seq(x).shape)         # -> torch.Size([8, 10])

### Step 4 — Re-initialize the weights with model.apply

`nn.Linear` already initializes sensibly, but you can override it. `model.apply(fn)` walks every submodule and calls `fn` on each, so we re-initialize only the `nn.Linear` layers (Kaiming-normal weights, zero biases). Checking that `fc1.bias` is all zeros afterward confirms the walk reached the layer.

In [ ]:
# Weight initialization: override the defaults if you want.
# (nn.Linear already inits sensibly; this shows how to control it.)
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
        nn.init.zeros_(m.bias)

model.apply(init_weights)                 # walks every submodule
print("after re-init, fc1.bias is all zeros:",
      bool(torch.all(model.fc1.bias == 0)))        # -> True

## Visualize the data & results

_Where do an MLP's parameters actually live? Per-layer trainable-parameter count for the MLP Linear(784->256) -> ReLU -> Linear(256->128) -> ReLU -> Linear(128->10), computed with the formula in*out + out._

### Step 1 — Count the parameters in each layer

Where do a network's parameters actually live? Each `nn.Linear(in, out)` holds a weight matrix (`in * out` numbers) plus one bias per output (`out`), so its parameter count is `in*out + out`; a `ReLU` has none. We walk a 784-256-128-10 MLP, recording the count for each layer (with the zero-parameter ReLUs in between), and confirm the total matches the 203530 from the model above.

In [ ]:
import numpy as np

# An MLP described as a list of (in_features, out_features) for each Linear,
# with parameter-free ReLU activations in between.
linears = [(784, 256), (256, 128), (128, 10)]

labels, values = [], []
for (in_f, out_f) in linears:
    params = in_f * out_f + out_f        # weight matrix (in*out) + bias (out)
    labels.append(f"Linear {in_f}->{out_f}")
    values.append(params)
    labels.append("ReLU")                # activation between Linear layers...
    values.append(0)                     # ...has zero parameters

labels, values = labels[:-1], values[:-1]   # drop trailing ReLU after last Linear

for lab, v in zip(labels, values):
    print(f"{lab:18s} {v:>8d}")
print("total parameters:", sum(values))   # -> 203530
# (matches torch:  sum(p.numel() for p in model.parameters()))

### Step 2 — Plot the per-layer parameter counts

A bar chart makes the imbalance obvious: the first `Linear 784->256` dominates the parameter budget, the ReLUs contribute nothing, and the small output layer is barely visible. Most of a network's weights live in its widest matmuls.

In [ ]:
import matplotlib.pyplot as plt

bar_colors = ["#4ea1ff", "#8b949e", "#4ea1ff", "#8b949e", "#7ee787"]
plt.bar(labels, values, color=bar_colors)
plt.ylabel("parameter count")
plt.title("Trainable parameters per layer of a 784-256-128-10 MLP")
plt.xticks(rotation=20)
plt.show()

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Type this in Colab. Define a minimal nn.Module subclass Net with one layer self.fc = nn.Linear(4, 2) declared in __init__ (call super().__init__() first) and a forward that returns self.fc(x). Instantiate it and print(model).

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Call super().__init__() as the first line of __init__. — _It sets up the bookkeeping that registers layers; skipping it raises an error._
- Declare the layer on self, then use it in forward. — _Only layers assigned to self are registered and trained._

**Answer:** import torch
import torch.nn as nn

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(4, 2)
    def forward(self, x):
        return self.fc(x)

model = Net()
print(model)
# Net(
#   (fc): Linear(in_features=4, out_features=2, bias=True)
# )

</details>

**Problem 2.** Type this in Colab. Reproduce the famous super().__init__() pitfall. Write a subclass that assigns self.fc = nn.Linear(4, 2) WITHOUT calling super().__init__() first. Instantiate it inside a try/except and print the error. Then fix it and confirm it constructs.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Omit super().__init__() and assign a layer to self. — _The registration machinery is not set up yet, so assigning a Module fails._
- Add super().__init__() as the first line to fix it. — _It initializes the parameter/submodule registries before any layer is assigned._

**Answer:** class Broken(nn.Module):
    def __init__(self):
        self.fc = nn.Linear(4, 2)     # no super().__init__() yet!
    def forward(self, x):
        return self.fc(x)

try:
    Broken()
except AttributeError as err:
    print("failed:", "Module.__init__()" in str(err))   # failed: True

class Fixed(nn.Module):
    def __init__(self):
        super().__init__()            # the fix
        self.fc = nn.Linear(4, 2)
    def forward(self, x):
        return self.fc(x)
print(Fixed())   # constructs fine

</details>

**Problem 3.** Type this in Colab. Build the canonical MLP Linear(784&rarr;256) &rarr; ReLU &rarr; Linear(256&rarr;10) as an nn.Module subclass (seed first with torch.manual_seed(0)). Run a forward pass on x = torch.randn(8, 784) and print the output shape. Predict the shape first.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Stack two nn.Linear layers with a ReLU between them. — _The hidden width 256 connects the layers; ReLU adds nonlinearity with no parameters._
- Call model(x) (never model.forward(x)). — _Calling the module runs hooks plus forward; a batch of 8 maps to (8, 10) logits._

**Answer:** torch.manual_seed(0)
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(256, 10)
    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

model = MLP()
x = torch.randn(8, 784)
logits = model(x)
print(logits.shape)     # torch.Size([8, 10])

</details>

**Problem 4.** Type this in Colab. Count the trainable parameters of that MLP two ways. Compute by hand with in*out + out per layer, then verify with sum(p.numel() for p in model.parameters()). Predict the total before running.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Apply in*out + out to each nn.Linear. — _A linear layer is a weight matrix (in*out) plus one bias per output (out)._
- Sum with p.numel() over model.parameters(). — _It iterates every registered weight tensor and totals their element counts._

**Answer:** # By hand, in*out + out per linear layer (ReLU has 0):
#   fc1: 784*256 + 256 = 200960
#   fc2: 256*10  + 10  =   2570
#   total              = 203530
n = sum(p.numel() for p in model.parameters())
print(n)                # 203530
print(n == 200960 + 2570)   # True

</details>

**Problem 5.** Type this in Colab. Show that a layer built inside forward is NOT registered. Define a model whose forward does h = nn.Linear(4, 4)(x) (a throwaway), with only self.out = nn.Linear(4, 2) in __init__. Print the names from model.named_parameters() and note the in-forward layer is absent.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- List [name for name, _ in model.named_parameters()]. — _Only layers assigned to self in __init__ appear; the in-forward layer never registers._
- Observe only out.* entries are present. — _The throwaway nn.Linear is rebuilt with new random weights each call and the optimizer never sees it._

**Answer:** class Leaky(nn.Module):
    def __init__(self):
        super().__init__()
        self.out = nn.Linear(4, 2)
    def forward(self, x):
        h = nn.Linear(4, 4)(x)      # throwaway -- NOT registered
        return self.out(h)

model = Leaky()
print([name for name, _ in model.named_parameters()])
# ['out.weight', 'out.bias']   -- the in-forward Linear is missing!

</details>

**Problem 6.** Type this in Colab. The same network with nn.Sequential. Build seq = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 10)), run it on x = torch.randn(8, 784), and confirm its parameter count matches the subclassed MLP (203530).

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Stack the layers in nn.Sequential. — _For a straight chain it glues layers without writing a class._
- Count with sum(p.numel() for p in seq.parameters()). — _Same layers mean the same registered parameters and identical count._

**Answer:** torch.manual_seed(0)
seq = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 10))
x = torch.randn(8, 784)
print(seq(x).shape)                                  # torch.Size([8, 10])
print(sum(p.numel() for p in seq.parameters()))      # 203530

</details>

**Problem 7.** Type this in Colab. Peek at the checkpoint keys and confirm parameters track gradients. For the subclassed MLP, print list(model.state_dict().keys()) and model.fc1.weight.requires_grad. Predict both before running.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Read model.state_dict().keys(). — _It names every registered weight tensor — what a checkpoint stores._
- Check model.fc1.weight.requires_grad. — _Registered parameters default to requires_grad=True so autograd fills their .grad._

**Answer:** print(list(model.state_dict().keys()))
# ['fc1.weight', 'fc1.bias', 'fc2.weight', 'fc2.bias']
print(model.fc1.weight.requires_grad)   # True

</details>

**Problem 8.** Type this in Colab. Reproduce a layer-size-mismatch error. Build nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(64, 2)) (note: 32 &ne; 64), run it on torch.randn(5, 10) inside a try/except, and print the error. Then fix the second layer's in_features to 32 and print the output shape.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Wire out_features=32 into in_features=64 and run. — _The forward matmul needs matching dims; a mismatch raises a shape error._
- Change the second layer to nn.Linear(32, 2). — _Each layer's out_features must equal the next layer's in_features._

**Answer:** bad = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(64, 2))
x = torch.randn(5, 10)
try:
    bad(x)
except RuntimeError as err:
    print("mismatch:", "cannot be multiplied" in str(err))   # mismatch: True

good = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 2))
print(good(x).shape)      # torch.Size([5, 2])

</details>